# 11 — Reference Φ integrator (partial) + the Step-7 / Phase-1 handoff

Two things live here: the last **partially built** Phase-0 node, and the
board of what Phase 1 inherits.

**The integrator** (`fluxes/integrate.py`, ADR 0006) is the reference
producer of (Φ, ΔY = νΦ) label pairs. Target A's φ is a **time-integrated
effective flux** per step, Φⱼ = ∫φⱼ dt — not an instantaneous rate — because
every label encodes a relaxation (even dt = 1e-6 s is stiff across the whole
box). The integrator augments the ODE state to [Y, Φ] and returns
Y := Y₀ + νΦ, so conservation is **exact by construction**.

It is 🟡 **partial**: hot strata (T9 ≥ 5) are censored by integrator cost
(the neglected tabular-EC ρYₑ Jacobian chain dominates the hot weak-drift
phase; the ADR-0006 contingency is the unlock), and the independent-leg
energy cross-check sits at 2.1 % median for mesa_80 against a 1 % band —
carried as an OPEN MARGIN item.

**The heavy cells are gated.** `compile_network` is a multi-minute
pynucastro build, so integration runs only under `NB_RUN_INTEGRATE=1`. The
handoff board always renders.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
if not (_here / "nbsupport.py").exists():
    _root = next(p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists())
    _here = _root / "notebooks" / "phase0"
sys.path.insert(0, str(_here))

import os
import time

import matplotlib.pyplot as plt
import numpy as np

import nbsupport as nbs

nbs.style()
QUICK = nbs.QUICK
NET = "mesa_80"
RUN_INTEGRATION = os.environ.get("NB_RUN_INTEGRATE", "0") == "1"
CL = nbs.parse_checklist()

In [ ]:
nbs.provenance_header(
    "11",
    "Reference Φ integrator (partial) + Step-7 handoff",
    nbs.status_of([11, 2, 3, 4, 5, 12, 13, 14, 15], checklist=CL),
    results_rows=[
        "2026-07-12: integrator correctness — returned Y := Y0 + νΦ ⇒ ΔY = νΦ, conservation EXACT by construction (solver-vs-identity residual 2.1e-16); BDF ≡ Radau ≡ rtol 1e-10 to 5e-8 rel",
        "2026-07-12: energy identity (gate as written) — flux route ΣQⱼΦⱼ vs mass-excess bookkeeping residual median 6e-6…2.5e-5, max 8.7e-4, frac ≤1 % = 1.000 in every MEASURED stratum, both nets — note the hot strata (T9 ≥ 5) are CENSORED, not measured (near-algebraic caveat: constant mass-derived Q)",
        "2026-07-12: independent-leg cross-check — median ratio 0.979 / 0.996 (mesa_80/151); mesa_80 at 2.1 % is OUTSIDE the 1 % band — OPEN MARGIN item carried to Phase 1",
        "2026-07-12: throughput — ≥383 / 1,519 s per nine-dt state; 17/36 and 10/24 states censored at 4,800 s, ALL censored are T9 ≥ 5 ⇒ corpus-scale local Φ INFEASIBLE (≥1.1e5 / 4.4e5 core-h), stratified T9 < 5 subsets feasible (10³–10⁴ states ≈ 4–40 core-days)",
        "2026-07-12: label agreement (T9 < 5) — QSE window 0.74–0.99 of species in the handshake band across the nine dts",
    ],
    data=[
        "data/zenodo/.../training_sets (labels)",
        "data/stoich/nu_mesa80.npz (ν, Q)",
        "docs/phase0-killtest-verdict.md §Step-7 handoff",
    ],
    scripts=["scripts/step6_integrate_check.py", "scripts/step6_eps_pin.py"],
)

## Part 1 — the integrator

Gated: set `NB_RUN_INTEGRATE=1` to compile the network (minutes) and
integrate one T9 < 5 state. Otherwise the measured results are quoted below.

In [ ]:
if not RUN_INTEGRATION:
    print(
        "Integration cells SKIPPED (compile_network is a multi-minute pynucastro build).\n"
        "Run with:  NB_RUN_INTEGRATE=1 uv run jupyter execute notebooks/phase0/11-integrator-and-handoff.ipynb\n"
        "or set the env var before launching jupyter lab.\n\n"
        "Measured results (RESULTS.md 2026-07-12; scripts/step6_integrate_check.py):\n"
        "  · solver-vs-identity residual        2.1e-16   (ΔY = νΦ exact by construction)\n"
        "  · energy identity, frac ≤ 1 %        1.000     (every MEASURED stratum; T9 ≥ 5 censored)\n"
        "  · energy residual median             6e-6 … 2.5e-5   (max 8.7e-4)\n"
        "  · independent-leg ratio              0.979 / 0.996   (mesa_80 at 2.1 % — OPEN)\n"
        "  · throughput                         ≥383 / 1,519 s per nine-dt state\n"
        "  · censored states                    17/36, 10/24 — ALL at T9 ≥ 5"
    )
else:
    from gnn_nucleo.data.labels import load_step_frame
    from gnn_nucleo.data.subsample import load_subsample_ids
    from gnn_nucleo.fluxes.compile import compile_network
    from gnn_nucleo.fluxes.integrate import integrate_state
    from gnn_nucleo.graph import load_isotope_table

    table = load_isotope_table(NET)
    A = table.A.astype(np.float64)
    mass = np.array([nuc.mass for nuc in table.nuclei])  # mass excesses, as in the script
    z = nbs.load_nu(NET)
    nu, Q = z["nu"], z["Q"]

    ids = load_subsample_ids(NET)
    cols = ["logT", "logRho"] + [f"initial_{x}" for x in table.names]
    df0 = load_step_frame(NET, "1e-6", columns=cols, state_ids=ids[:4000])
    t9 = 10.0 ** df0["logT"].to_numpy() / 1e9
    cool = np.nonzero((t9 > 3.3) & (t9 < 4.5))[0][:1]  # QSE window, T9 < 5 (feasible strata)
    s = int(cool[0])
    Xi = df0[[f"initial_{x}" for x in table.names]].to_numpy()[s]
    T = 10.0 ** float(df0["logT"].iloc[s])
    rho = 10.0 ** float(df0["logRho"].iloc[s])
    print(f"compiling {NET} (pynucastro build — minutes) ...")
    t0 = time.time()
    cn = compile_network(NET)
    print(f"  compiled in {time.time() - t0:.0f} s")
    print(f"integrating state {ids[s]} at T9 = {T / 1e9:.2f}, rho = {rho:.2e}, dt = 1e-3 s ...")
    t0 = time.time()
    res = integrate_state(cn, T, rho, Xi / A, 1e-3)
    print(f"  done in {time.time() - t0:.1f} s | success={res.success} nfev={res.nfev} "
          f"| identity_resid = {res.identity_resid:.2e}")

### Figure 1 — conservation is exact by construction, not by tolerance

The integrator returns Y := Y₀ + νΦ. The conserved sums are therefore exact
*whatever the solver did*; `identity_resid` measures how far the solver's own
Y drifted from the returned identity (measured 2.1e-16 — float64 rounding).

In [ ]:
if RUN_INTEGRATION:
    dY = res.Y - Xi / A
    dY_from_flux = nu @ res.Phi
    baryon = float(np.abs(A @ dY))
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
    ax = axes[0]
    ax.semilogy(np.maximum(np.abs(dY - dY_from_flux), 1e-24), ".", ms=4)
    ax.set_xlabel("species index")
    ax.set_ylabel("|ΔY − νΦ|")
    ax.set_title(f"ΔY = νΦ holds identically (max {np.abs(dY - dY_from_flux).max():.1e})")
    ax = axes[1]
    e_flux = float(Q @ res.Phi)
    e_comp = -float(mass @ dY)
    resid = abs(e_flux - e_comp) / abs(e_comp) if e_comp else np.nan
    ax.bar(["flux route\nΣQⱼΦⱼ", "composition route\n−Σ mᵢ ΔYᵢ"], [e_flux, e_comp],
           color=["#0072B2", "#009E73"])
    ax.set_ylabel("energy [MeV per baryon-ish units]")
    ax.set_title(f"Energy identity — relative residual {resid:.2e} (gate ≤ 1e-2)")
    print(f"baryon drift |Σ Aᵢ dYᵢ| = {baryon:.2e} | solver-vs-identity = {res.identity_resid:.2e}")
    nbs.caption(
        fig,
        "Quick-look on one T9 < 5 state. Conservation holds to float64 rounding because the "
        "returned Y IS Y0 + νΦ — the property the whole Target-A design rests on. The energy "
        "identity passes, but note the measured caveat: with constant mass-derived Q this "
        "comparison is near-algebraic (it bounds Q-table rounding, not route physics), and Qⱼ(T) "
        "corrections are not modeled.",
        results=[
            "RESULTS.md 2026-07-12 integrator-correctness rows (identity residual 2.1e-16)",
            "RESULTS.md 2026-07-12 energy-identity rows (frac ≤1 % = 1.000; near-algebraic caveat)",
        ],
        scripts=["scripts/step6_integrate_check.py"],
    )
else:
    print("skipped — see the quoted measurements above (NB_RUN_INTEGRATE=1 to run)")

### Figure 2 — why the node is only PARTIAL: the cost wall

Throughput is measured, and it decides the Phase-1 supervision plan. All
censored states are T9 ≥ 5: the neglected tabular-EC ρYₑ Jacobian chain
dominates the hot weak-drift phase.

In [ ]:
# Measured throughput/feasibility (RESULTS.md 2026-07-12; docs/phase0-killtest-verdict.md).
FEAS = [
    ("corpus-scale local Φ\n(2²⁰ states/net)", 1.1e5, 4.4e5, "INFEASIBLE"),
    ("stratified T9 < 5 subset\n(10³–10⁴ states)", 4 * 24, 40 * 24, "FEASIBLE"),
]
fig, ax = plt.subplots(figsize=(9.5, 4.2))
y = np.arange(len(FEAS))
for i, (label, lo, hi, verdict) in enumerate(FEAS):
    ax.barh(i, hi, left=0, color="#D55E00" if verdict == "INFEASIBLE" else "#009E73", alpha=0.85)
    ax.text(hi * 1.2, i, f"{verdict} — {lo:.0f}…{hi:.0f} core-h", va="center", fontsize=9,
            color="#D55E00" if verdict == "INFEASIBLE" else "#005f45", fontweight="bold")
ax.set_yticks(y, [f[0] for f in FEAS], fontsize=8.5)
ax.set_xscale("log")
ax.set_xlim(1, 1e7)
ax.set_xlabel("core-hours (log scale)")
ax.set_title("Local Φ-label generation: what is affordable")
ax.axvline(54, color="#0072B2", ls="--", lw=1.2)
ax.text(58, -0.42, "the whole bbq rerun\ncampaign was 54 core-h", fontsize=7.5, color="#0072B2")
nbs.caption(
    fig,
    "Measured at ≥383 / 1,519 s per nine-dt state (mesa_80/151; censored lower bounds), 17/36 and "
    "10/24 states censored at the 4,800 s deadline — ALL of them T9 ≥ 5. Hence the Phase-1 "
    "supervision plan: shipped ΔX as primary supervision, local Φ as AUXILIARY labels restricted "
    "to T9 < 5 strata where generation is proven. Hot-strata Φ labels need the ADR-0006 Yₑ-chain "
    "Jacobian first — and their rate config awaits the benchmark-vs-physics decision.",
    results=[
        "RESULTS.md 2026-07-12 throughput rows (≥383/1,519 s per nine-dt state; censored all T9 ≥ 5)",
        "docs/phase0-killtest-verdict.md §Step-7 handoff (supervision plan)",
    ],
    scripts=["scripts/step6_integrate_check.py"],
)

## Part 2 — the Step-7 / Phase-1 handoff board

What Phase 1 inherits, straight from the checklist parser: rows 2–5 and
12–15 are the model-dependent measurements that cannot be made until there
is a model. **Row 12 is the pass/fail gate and the biggest single lever** —
the whole per-step |ΔYₑ| ≲ 3e-6 budget rests on an *assumed* systematic
accumulation model until that slope is measured.

In [ ]:
OPEN_ROWS = [2, 3, 4, 5, 12, 13, 14, 15]
fig, ax = plt.subplots(figsize=(13.5, 6.4))
ax.axis("off")
for i, r in enumerate(OPEN_ROWS):
    y = len(OPEN_ROWS) - i
    done = CL["rows"].get(r)
    meas, gate = CL["rowdata"].get(r, ("(row not parsed)", ""))
    meas = meas if len(meas) < 92 else meas[:89] + "…"
    gate = gate if len(gate) < 62 else gate[:59] + "…"
    color = "#009E73" if done else ("#BBBBBB" if done is False else "#FFFFFF")
    ax.add_patch(plt.Rectangle((0, y - 0.44), 13.4, 0.88, fc="#F7F7F7", ec="#DDDDDD"))
    ax.add_patch(plt.Rectangle((0.08, y - 0.3), 0.42, 0.6, fc=color, ec="#333333", lw=0.8))
    ax.text(0.29, y, str(r), va="center", ha="center", fontsize=8.5, fontweight="bold")
    ax.text(0.65, y + 0.16, meas, va="center", fontsize=7.6)
    ax.text(0.65, y - 0.17, f"gate: {gate}", va="center", fontsize=7, color="#666666",
            style="italic")
ax.set_xlim(0, 13.5)
ax.set_ylim(0.3, len(OPEN_ROWS) + 1.1)
ax.text(0.08, len(OPEN_ROWS) + 0.75,
        f"Phase-1 open measurements — {sum(1 for r in OPEN_ROWS if not CL['rows'].get(r))}"
        f"/{len(OPEN_ROWS)} still open (parsed live from docs/phase0-checklist.md)",
        fontsize=11.5, fontweight="bold")
ax.text(0.08, len(OPEN_ROWS) + 0.42,
        "All require a trained model — no feature pipeline, GNN, or training loop exists yet.",
        fontsize=8.5, color="#666666")
nbs.caption(
    fig,
    "Row 12 (Yₑ-residual accumulation slope) is THE pass/fail gate: slope ≈ 1 (systematic) keeps "
    "or tightens the 3e-6 per-step budget; slope ≈ 0.5 (random walk) may relax it toward ~5e-5 — "
    "two orders of magnitude of engineering difficulty ride on it. Rows 3/4 shape the architecture "
    "(channel ablations, weight-shared vs untied); row 5 tests the size-transfer claim; row 15 the "
    "temporal governor — whose design premises changed when the '6–8 orders' timescale separation "
    "was retired by measurement.",
    results=[
        "docs/phase0-checklist.md rows 2–5, 12–15 (parsed live)",
        "docs/phase0-killtest-verdict.md §Step-7 handoff",
    ],
    scripts=["(no producers yet — these are Phase-1 measurements)"],
)

## What Phase 1 starts with (from `docs/phase0-killtest-verdict.md`)

- **Flux head dims**: 607 / 1,518 columns (mesa_80/151); net view via
  `pair_col` (327 / 846 net columns). Decode = fixed ν ⇒ conservation by
  construction. Weak sector structural, never masked.
- **Mask parameters**: NONE (measured empty at every ε, both boundary
  variants). Carry κ diagnostics as **features**, not masks.
- **Supervision**: shipped ΔX primary; Φ auxiliary from
  `fluxes/integrate.py` restricted to T9 < 5 (QSE window first).
- **Loss-weight targets**: top-|dẎₑ| channels (⁵⁶Ni EC, ³¹S EC, ⁵²Fe EC,
  p EC) + the finalized bridge sets (notebook 06).
- **Target-B fallback triggers carried into Phase 1** (ADR 0001):
  cond(S_active) > 1e6 on any future masked configuration; >~90 % of
  net-flux-carrying reactions below the Yₑ floor on validation data; or
  Target B matching A within 2× |dYₑ| while being materially simpler.

### Blocked on human decisions (STEP6_REPORT §6)

1. **Benchmark-vs-physics fork at T9 ≳ 5** — train on shipped
   (bug-displaced) labels or corrected physics? Affects supervision,
   evaluation, and every NNN-comparison claim.
2. **External communication** — the label pathology implicates the published
   NNN training sets (notebook 09).
3. **GPU fleet** — becomes blocking at the Step-7 ablations.

## What this notebook does NOT show

- Any model: none exists yet (that is the point of the board).
- The label-agreement bands (0.74–0.99 of species in-band, T9 < 5) and the
  BDF≡Radau method cross-check — `scripts/step6_integrate_check.py` stdout.